In [ ]:
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

from functools import partial
from itertools import batched, chain
from math import ceil
from typing import Literal

import numpy as np
from astropy import units as u
from astropy.coordinates import ICRS, SkyCoord
from astropy.table import QTable
from astropy_healpix import HEALPix
from ligo.skymap import plot  # noqa: F401
from ligo.skymap.util import progress_map_vectorized
from m4opt.fov import footprint_healpix
from m4opt.missions import uvex as mission
from matplotlib import colors
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from survey import survey_programs
from tqdm.auto import tqdm


def movie_figure(
    pixel_height: Literal[720, 1080, 1440, 2160, 4320] = 1440, aspect=16 / 9
):
    """Create a Matplotlib figure for a common HD video format."""
    _, fig_height = plt.rcParams["figure.figsize"]
    return plt.figure(
        figsize=(fig_height * aspect, fig_height), dpi=pixel_height / fig_height
    )

In [ ]:
plan = QTable.read("../tables/plan.ecsv")
obs = plan[plan["action"] == "observe"]
hpx = HEALPix(nside=1024, frame=ICRS())

footprints = progress_map_vectorized(
    partial(footprint_healpix, hpx, mission.fov),
    obs["target_coord"].unmasked,
    obs["roll"].unmasked,
    jobs=None,
)

In [ ]:
survey_footprints = [
    np.arange(hpx.npix)
    if program.region is None
    else footprint_healpix(hpx, program.region)
    for program in survey_programs
]

In [ ]:
duration = 15
"""Animation duration in seconds."""

fps = 30
"""Frame rate in frames per second."""

frames = duration * fps + 1
"""Number of frames, including initial blank frame."""

dwells_per_frame = ceil(len(footprints) / (frames - 1))


def get_visit_map(footprints):
    return np.bincount(np.concatenate(footprints), minlength=hpx.npix)

In [ ]:
fig = movie_figure(2160)
gs = plt.GridSpec(2, 1, height_ratios=(3, 1))


cumulative_visit_map = np.zeros(hpx.npix, dtype=np.intp)
survey_completion = np.empty((frames, len(survey_footprints)))
survey_completion[0] = 0
survey_completion[1:] = np.nan

ax_img = fig.add_subplot(
    gs[0], projection="astro aitoff", center=SkyCoord(8 * u.hourangle, 0 * u.deg)
)
norm = colors.LogNorm(vmin=0.8, vmax=get_visit_map(footprints).max(), clip=True)
im_old = ax_img.imshow_hpx(cumulative_visit_map, norm=norm)
fig.colorbar(im_old).set_label("Number of visits")
ax_img.grid()

ax_timeline = fig.add_subplot(gs[1])
ax_timeline.set_xlim(0, (frames - 1) * dwells_per_frame)
ax_timeline.set_ylim(0, 1)
ax_timeline.set_xlabel("Dwells")
ax_timeline.set_ylabel("Completion")
lines = ax_timeline.plot(
    np.arange(frames) * dwells_per_frame,
    survey_completion,
    label=[program.name for program in survey_programs],
)
ax_timeline.legend(loc="upper right")


def func(i_batch):
    i, batch = i_batch
    global im_old, cumulative_visit_map
    im_old.remove()

    cumulative_visit_map += get_visit_map(batch)
    survey_completion[i] = [
        np.minimum(cumulative_visit_map[footprint] / program.visits, 1).sum()
        / len(footprint)
        for footprint, program in zip(survey_footprints, survey_programs)
    ]
    for line, ydata in zip(lines, survey_completion.T):
        line.set_ydata(ydata)
    im = ax_img.imshow_hpx(cumulative_visit_map, norm=norm)

    result = [im_old, im, *lines]
    im_old = im
    return result


with tqdm(total=frames, leave=False) as progress:

    def progress_callback(frame_number, _):
        progress.n = frame_number + 1
        progress.refresh()

    FuncAnimation(
        fig,
        func,
        enumerate(
            chain(
                [[np.zeros(0, dtype=np.intp)]],
                batched(footprints, dwells_per_frame),
            )
        ),
        save_count=frames,
        interval=1000 / fps,
    ).save("../visualizations/visit-map.mp4", progress_callback=progress_callback)